# Kafka 交易事件生产

构造交易事件并发送到 Kafka。先运行本地示例，再连接自己的 Kafka 服务。

## 运行条件

需要安装 `confluent-kafka`，并准备可访问的 Kafka 和目标 Topic。将 `send_to_kafka` 改为 `True` 才发送消息；重复运行会重复发送。发送结束后检查消费者是否收到两条事件。


In [ ]:
from __future__ import annotations
import argparse
import json
import time
from dataclasses import asdict, dataclass
from datetime import datetime


## 实现


In [ ]:
@dataclass(frozen=True)
class TradeEvent:
    event_type: str
    event_time: str
    order_id: str
    user_id: str
    shop_id: str
    city: str
    status: str
    amount: float


In [ ]:
def build_producer(bootstrap_servers: str):
    from confluent_kafka import Producer

    return Producer(
        {
            "bootstrap.servers": bootstrap_servers,
            "acks": "all",
            "retries": 5,
            "linger.ms": 20,
            "compression.type": "zstd",
            "client.id": "processing-demo-producer",
        }
    )


In [ ]:
def send_events(bootstrap_servers: str, topic: str, events: list[TradeEvent]) -> None:
    producer = build_producer(bootstrap_servers)
    for event in events:
        producer.produce(
            topic,
            key=event.order_id.encode("utf-8"),
            value=json.dumps(asdict(event), ensure_ascii=False).encode("utf-8"),
        )
        producer.poll(0)
    remaining = producer.flush(timeout=10)
    if remaining:
        raise RuntimeError(f"仍有 {remaining} 条消息未发送完成")


In [ ]:
def build_sample_events() -> list[TradeEvent]:
    event_time = datetime.fromtimestamp(time.time()).strftime("%Y-%m-%d %H:%M:%S")
    return [
        TradeEvent(
            event_type="order_paid",
            event_time=event_time,
            order_id="O202605090001",
            user_id="U7788",
            shop_id="S1008",
            city="Shanghai",
            status="PAY_SUCCESS",
            amount=3580.0,
        ),
        TradeEvent(
            event_type="payment_result",
            event_time=event_time,
            order_id="O202605090002",
            user_id="U8821",
            shop_id="S1008",
            city="Beijing",
            status="PAY_FAILED",
            amount=5200.0,
        ),
    ]


## 本地检查

此单元不连接外部服务。


In [ ]:
events = build_sample_events()
[asdict(event) for event in events]


## 连接服务

先修改地址与参数，再开启执行开关。


In [ ]:
bootstrap_servers = "localhost:9092"
topic = "trade_events"
send_to_kafka = False

if send_to_kafka:
    send_events(bootstrap_servers, topic, events)


## 命令行入口

导出为脚本后使用；在 Notebook 内运行仅定义入口，不解析内核参数。


In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--bootstrap", required=True)
    parser.add_argument("--topic", default="trade_events")
    args = parser.parse_args()

    send_events(args.bootstrap, args.topic, build_sample_events())


if __name__ == "__main__" and "get_ipython" not in globals():
    main()
